# GFS Research Pipeline for Kalshi NYC Temperature Markets

We build a custom dataset that uses the GFS forecasts and raw atomospheric data found in NOMADS to find error signals 
in comparison to actual central park NYC tempeartures on NOAA

**Target**: Predict when GFS forecast errors occur  
**Dataset**: One row per (init_date, market_date)

In [ ]:
# Imports
from gfs_research_pipeline import build_research_dataset
from noaa_actuals import fetch_central_park_actuals, fetch_recent_actuals
from datetime import datetime, timedelta
import pandas as pd

# Display settings - show ALL rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Fetch Actual Central Park Temperatures

Using NOAA GHCN Daily API (token loaded from `.env` file)

In [ ]:
# Fetch recent actuals from NOAA (note: ~1 week data lag)
actuals_df = fetch_recent_actuals(days=30)

print(f"Fetched {len(actuals_df)} days of actual temperatures")
print()
print(actuals_df.to_string())

Fetching 2026-02-23 to 2026-03-25...
  Retrieved 48 records
Fetched 24 days of actual temperatures

          date  actual_high_celsius  actual_low_celsius
0   2026-02-23                 1.70               -2.20
1   2026-02-24                -0.60               -5.00
2   2026-02-25                 6.70               -3.30
3   2026-02-26                 9.40                1.70
4   2026-02-27                 6.70               -1.70
5   2026-02-28                12.20                0.60
6   2026-03-01                 6.10               -1.10
7   2026-03-02                 0.60               -6.10
8   2026-03-03                 2.20               -1.70
9   2026-03-04                 9.40                1.70
10  2026-03-05                 7.80                2.80
11  2026-03-06                 5.00                2.80
12  2026-03-07                11.10                2.20
13  2026-03-08                20.60               10.00
14  2026-03-09                22.80               10.60
15  

## 2. Build GFS Research Dataset

Downloads GFS data, extracts features, merges with actuals

In [ ]:
# Define GFS initialization dates to process
# NOMADS has ~10 days of data, so we go back 3-9 days
init_dates = [datetime.now() - timedelta(days=i) for i in range(3, 10)]

print(f"Processing {len(init_dates)} GFS initializations...")
print("=" * 60)

Processing 7 GFS initializations...


In [ ]:
# Build the research dataset
df = build_research_dataset(init_dates, actuals_df)

print(f"\nDataset built: {len(df)} rows")

Processing 2026-03-23 00Z... High: 8.0°C, Low: 0.5°C
Processing 2026-03-22 00Z... High: 11.8°C, Low: 2.8°C
Processing 2026-03-21 00Z... High: 18.4°C, Low: 7.3°C
Processing 2026-03-20 00Z... High: 14.1°C, Low: 7.3°C
Processing 2026-03-19 00Z... High: 14.4°C, Low: 4.5°C
Processing 2026-03-18 00Z... High: 5.2°C, Low: 0.3°C
Processing 2026-03-17 00Z... High: 2.8°C, Low: -2.2°C

Dataset built: 7 rows


## 3. View the Dataset

In [ ]:
# View FULL dataset - all rows, all columns
print("COMPLETE DATASET")
print("=" * 100)
print(df.to_string())

COMPLETE DATASET
    init_date init_cycle market_date  n_forecast_hours  forecasted_high_celsius  forecasted_low_celsius  mean_lead_time_hours  mean_u10_mps  mean_v10_mps  mean_mslp_hpa  mean_t850_celsius  mean_z500_meters  mean_wind_speed_mps  mean_wind_direction_deg  mean_temp_spread_surface_850  onshore_wind_fraction  month  day_of_year  is_winter  is_summer wind_regime  actual_high_celsius  actual_low_celsius  error_high_celsius  error_low_celsius
0  2026-03-23         00  2026-03-24                 8                     7.98                    0.50                 40.50          1.10         -0.89        1026.51              -7.29           5501.04                 4.50                   244.37                         11.49                   0.12      3           83          0          0       light                  NaN                 NaN                 NaN                NaN
1  2026-03-22         00  2026-03-23                 8                    11.81                    2.75  

In [ ]:
# FORECASTS VS ACTUALS
print("FORECASTS VS ACTUALS")
print("=" * 80)
forecast_cols = ['init_date', 'market_date', 
                 'forecasted_high_celsius', 'actual_high_celsius', 'error_high_celsius',
                 'forecasted_low_celsius', 'actual_low_celsius', 'error_low_celsius']
df_forecasts = df[[c for c in forecast_cols if c in df.columns]].copy()
print(df_forecasts.to_string())

FORECASTS VS ACTUALS
    init_date market_date  forecasted_high_celsius  actual_high_celsius  error_high_celsius  forecasted_low_celsius  actual_low_celsius  error_low_celsius
0  2026-03-23  2026-03-24                     7.98                  NaN                 NaN                    0.50                 NaN                NaN
1  2026-03-22  2026-03-23                    11.81                  NaN                 NaN                    2.75                 NaN                NaN
2  2026-03-21  2026-03-22                    18.36                  NaN                 NaN                    7.25                 NaN                NaN
3  2026-03-20  2026-03-21                    14.07                  NaN                 NaN                    7.27                 NaN                NaN
4  2026-03-19  2026-03-20                    14.39                  NaN                 NaN                    4.45                 NaN                NaN
5  2026-03-18  2026-03-19                     5.2

In [ ]:
# RAW ATMOSPHERIC FEATURES
print("RAW ATMOSPHERIC FEATURES")
print("=" * 80)
raw_cols = ['market_date', 'mean_u10_mps', 'mean_v10_mps', 'mean_mslp_hpa', 
            'mean_t850_celsius', 'mean_z500_meters']
df_raw = df[[c for c in raw_cols if c in df.columns]].copy()
print(df_raw.to_string())

RAW ATMOSPHERIC FEATURES
  market_date  mean_u10_mps  mean_v10_mps  mean_mslp_hpa  mean_t850_celsius  mean_z500_meters
0  2026-03-24          1.10         -0.89        1026.51              -7.29           5501.04
1  2026-03-23          0.90         -4.76        1012.43              -1.59           5507.39
2  2026-03-22          1.29          2.68        1007.09               7.87           5604.46
3  2026-03-21          1.62         -1.71        1010.26               2.78           5535.65
4  2026-03-20          1.35          3.27        1015.31               1.51           5557.92
5  2026-03-19         -0.83          3.31        1025.15              -6.17           5526.72
6  2026-03-18          1.55          0.22        1027.51             -13.04           5467.76


In [ ]:
# ENGINEERED FEATURES
print("ENGINEERED FEATURES")
print("=" * 80)
eng_cols = ['market_date', 'mean_wind_speed_mps', 'mean_wind_direction_deg',
            'mean_temp_spread_surface_850', 'onshore_wind_fraction', 'wind_regime']
df_engineered = df[[c for c in eng_cols if c in df.columns]].copy()
print(df_engineered.to_string())

ENGINEERED FEATURES
  market_date  mean_wind_speed_mps  mean_wind_direction_deg  mean_temp_spread_surface_850  onshore_wind_fraction wind_regime
0  2026-03-24                 4.50                   244.37                         11.49                   0.12       light
1  2026-03-23                 5.25                   173.48                          8.13                   0.00    moderate
2  2026-03-22                 3.64                   214.73                          4.43                   0.12       light
3  2026-03-21                 3.77                   254.13                          8.03                   0.12       light
4  2026-03-20                 3.98                   214.78                          7.13                   0.12       light
5  2026-03-19                 3.60                   168.88                          9.23                   0.88       light
6  2026-03-18                 3.75                   255.41                         13.23                

In [ ]:
# TEMPORAL FEATURES
print("TEMPORAL FEATURES")
print("=" * 80)
temp_cols = ['market_date', 'month', 'day_of_year', 'is_winter', 'is_summer', 'mean_lead_time_hours']
df_temporal = df[[c for c in temp_cols if c in df.columns]].copy()
print(df_temporal.to_string())

TEMPORAL FEATURES
  market_date  month  day_of_year  is_winter  is_summer  mean_lead_time_hours
0  2026-03-24      3           83          0          0                 40.50
1  2026-03-23      3           82          0          0                 40.50
2  2026-03-22      3           81          0          0                 40.50
3  2026-03-21      3           80          0          0                 40.50
4  2026-03-20      3           79          0          0                 40.50
5  2026-03-19      3           78          0          0                 40.50
6  2026-03-18      3           77          0          0                 40.50


In [ ]:
# View single row transposed (easier to see all columns)
print("\nSINGLE ROW DETAIL (transposed view)")
print("=" * 50)
print(df.iloc[0].to_string())


SINGLE ROW DETAIL (transposed view)
init_date                       2026-03-23
init_cycle                              00
market_date                     2026-03-24
n_forecast_hours                         8
forecasted_high_celsius               7.98
forecasted_low_celsius                0.50
mean_lead_time_hours                 40.50
mean_u10_mps                          1.10
mean_v10_mps                         -0.89
mean_mslp_hpa                      1026.51
mean_t850_celsius                    -7.29
mean_z500_meters                   5501.04
mean_wind_speed_mps                   4.50
mean_wind_direction_deg             244.37
mean_temp_spread_surface_850         11.49
onshore_wind_fraction                 0.12
month                                    3
day_of_year                             83
is_winter                                0
is_summer                                0
wind_regime                          light
actual_high_celsius                    NaN
actual_low_celsiu

## 4. Save Dataset

In [ ]:
# Save to CSV
output_path = "data/research_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to data/research_dataset.csv


## 5. More Historical Data (AWS Open Data)

GFS data in NOMAD only allows for past max 10 days. For more data, we use AWS Open Data which has GFS archives from ~2021 onwards.

**Warning**: AWS files are full global files (~30-50 MB each). For a month of data, expect ~10-20 GB of downloads.

In [ ]:
# Example: Build historical dataset from AWS
# Uncomment to run (will download large files)

# from gfs_historical import build_historical_dataset
# 
# # Fetch actuals for the historical period
# historical_actuals = fetch_central_park_actuals("2024-06-01", "2024-06-30")
# 
# # Build dataset from AWS (downloads ~30-50 MB per day)
# df_historical = build_historical_dataset(
#     start_date=datetime(2024, 6, 1),
#     end_date=datetime(2024, 6, 30),
#     actuals_df=historical_actuals
# )
# 
# print(df_historical)

## 6. Your Analysis

Add your own signal analysis below...

In [ ]:
# Your analysis here


## Dataset Schema

```
IDENTIFIERS:
  init_date                    - GFS initialization date
  init_cycle                   - Forecast cycle ('00')
  market_date                  - NYC local date being forecasted

TARGETS:
  forecasted_high_celsius      - GFS max 2m temp over NYC day
  forecasted_low_celsius       - GFS min 2m temp over NYC day
  actual_high_celsius          - Actual Central Park daily high
  actual_low_celsius           - Actual Central Park daily low
  error_high_celsius           - actual - forecast (+ = GFS cold)
  error_low_celsius            - actual - forecast (+ = GFS cold)

RAW ATMOSPHERIC FEATURES:
  mean_u10_mps                 - Eastward wind component
  mean_v10_mps                 - Northward wind component
  mean_mslp_hpa                - Mean sea level pressure
  mean_t850_celsius            - 850mb temperature
  mean_z500_meters             - 500mb geopotential height

ENGINEERED FEATURES:
  mean_wind_speed_mps          - Wind speed magnitude
  mean_wind_direction_deg      - Meteorological direction (FROM)
  mean_temp_spread_surface_850 - T2m - T850 (stability)
  onshore_wind_fraction        - Fraction with ocean wind
  wind_regime                  - calm/light/moderate/strong

TEMPORAL FEATURES:
  month, day_of_year, is_winter, is_summer, mean_lead_time_hours
```